*[cell 0]*

# LVL_2 Sweep Report — calibration on all data (a-priori gold standard)

The **calibration** is on all data (the sweep's `--condition all`: every condition of the LVL_1 handoff): this report
picks the configurations carried into the LVL2 run (`block_detection_and_removal_LVL2_all_rep.ipynb`), where the
hypotheses are tested. It reads a finished sweep only; it computes nothing that the sweep stages compute. The one file
it may write into the sweep folder is `_script_lvl2.py`, the byte copy of the `lvl2.py` the sweep ran: only when the
copy is missing and `lvl2.py` still has the sweep's fingerprint (`[cell 1]`).

**Designs.** The design key `dynamic` means the blocks of the analysis method: a change-point detector's blocks in
dynamic segmentation, the data's own blocks (method `Predefined`) in predefined segmentation. `control` (the matched
fixed-size blocks) is a design in dynamic segmentation only, and `session` only when a participant has 2 or more
sessions (otherwise the per-session unit fit is reported as `participant`). The report takes the designs and their
labels from the a-priori run's record (`apriori/inattent999000_heuristic.json`, §0b) and shows those present.

**Inputs.** The sweep folder written by `launch_LV2_sweep_random.py` with all three stages: the grid (`cluster*.csv`,
`_h2.csv`, `_flagged.csv`, `sweep_results.csv`), `--stage apriori` (`apriori/`: the behavioural heuristic of
`LVL_2_CONFIG`, the a-priori gold standard) and `--stage stability` (`stability/`: the bootstrap refits of the
calibration candidates).

**Gold standard.** An a-priori definition of careless responding (`inattent_behav`, §0b): the pattern-marker
thresholds (the ALL 99th column of the LVL_0 marker table, from the per-session permutation nulls), the
implausible-speed floor and the block rule (a block is flagged when more than `inattent_behav_block_percent_thresh` %
of its trials are marked; README, "Calibration"). Cluster and GMM are each calibrated to the configuration whose
flagged `dynamic` blocks overlap that standard most (Dice, DSC); exact ties are broken by cluster stability (median
bootstrap ARI), and between the two arms of a family the more stable pick is carried. In every k-means and GMM fit the
fastest component (lowest mean `adj_rt_level_block`) is the inattentive one.

**Outputs** (in `LVL_OUTS`, named `<sweep>_report_<table>.csv`): the DSC across the sweep (`dsc_summary`, `dsc_units`,
`dsc_variance`, `dsc_picks`, `dsc_maxima`, `dsc_configs`, and in qualitative bands, `dsc_bands`), the stability of the
candidates (`stability`), the dose-response fits and AUC contrasts (`dose_fits`; `dose_contrasts`, resampling distinct
solutions, and `dose_contrasts_per_config`, resampling configurations; `dose_picks`, `dose_points`), and
`<sweep>_selection.json`, the carried arms with their `LVL_2_CONFIG` values, `LVL_2_CONFIG_full` (every key of the
script copy's `LVL_2_CONFIG`, with those values) and `script_copy` (the path of the script copy). The final report
prints the tables; the LVL2 notebook checks its configuration against the selection. Section 4 draws the calibration
figures.

In [ ]:
# [cell 1]
# ===== INPUTS: set these before running =====
SWEEP_DIRS = None     # list of sweep folders in SWEEPS_DIR for ONE analysis method (its main grid plus any supplementary
                      # exponent grids), e.g. ["_sweep_lvl2_all_PELT_Attention_comp"]
LVL1_HANDOFF = None   # the LVL1 output prefix the sweep ran on, e.g. "lv1_out_20261005_120000"
# The absolute project folder holding code/, dset/ and dervatives/ (README, "Paths"; EMST /home/sjsuss626/EMST_files/sjs_dissertation)
PROJECT_ROOT = "SET_ME"
# The launcher's apriori and stability stages write apriori/ and stability/ inside the first sweep folder; this report
# reads them and writes nothing into the sweep folders but the script copy _script_lvl2.py when it is missing (below).

import ast
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from statsmodels.stats.multitest import multipletests

# ---- settings check (README, "Settings") ----------------
_SET_ME = {"PROJECT_ROOT": (PROJECT_ROOT, 'the absolute project folder holding code/, dset/ and dervatives/ '
                                          '(README, "Paths"); EMST /home/sjsuss626/EMST_files/sjs_dissertation')}
# name: (value, what to set)
_unset = [k for k, (v, _) in _SET_ME.items() if isinstance(v, str) and v == "SET_ME"]
if _unset:
    raise SystemExit(f'settings check (LVL2_sweep_report.ipynb): {len(_unset)} setting(s) still hold "SET_ME". Set each, '
                     'then run again (README, "Settings"):\n' + "\n".join(f"  - {k}  ([cell 1])\n      {_SET_ME[k][1]}"
                                                                          for k in _unset))
if not Path(PROJECT_ROOT).is_absolute():
    raise SystemExit(f'PROJECT_ROOT = {PROJECT_ROOT!r} is not an absolute path ([cell 1]; README, "Paths")')

# ---- locations: everything is inside PROJECT_ROOT ----------------
_OUT = Path(PROJECT_ROOT) / "dervatives" / "practical_application_generic"
LVL_OUTS = _OUT / "lvl_outs"                                               # the LVL_1 handoff
SWEEPS_DIR = _OUT / "sweeps"                                               # the sweep folders
CODE_DIR = Path(PROJECT_ROOT) / "code" / "practical_application_generic"   # lvl2.py and the LVL2 notebook

import sys
_dwb, sys.dont_write_bytecode = sys.dont_write_bytecode, True   # as python -B: no __pycache__/
from sweep_provenance import read_sweep, pipeline_fingerprint, grid_fingerprint
sys.dont_write_bytecode = _dwb

if not SWEEP_DIRS or LVL1_HANDOFF is None:
    raise SystemExit("set SWEEP_DIRS (a list of LVL_2 sweep folders) and LVL1_HANDOFF at the top of this cell")
SWEEP_DIRS = [SWEEPS_DIR / d for d in SWEEP_DIRS]
CACHE_DIR  = SWEEP_DIRS[0]                 # home of the stages' apriori/ and stability/
L1_PREFIX  = LVL_OUTS / str(LVL1_HANDOFF).removesuffix("_long.csv")
_l1_missing = [s for s in ("_long.csv", "_features.csv", "_results.csv", "_meta.json")
               if not Path(f"{L1_PREFIX}{s}").is_file()]
if not re.match(r"lv1_out_\d{8}_\d{6}", L1_PREFIX.name) or _l1_missing:
    raise SystemExit(f"LVL1_HANDOFF = {LVL1_HANDOFF!r} is not a complete LVL_1 handoff "
                     f"(missing: {_l1_missing})")
# The handoff's record (_meta.json, LVL_1 [cell 25]): its segmentation, "dynamic" (a detector's blocks with their matched
# fixed-size control) or "predefined" (the data's own blocks, method Predefined, no control).
with open(f"{L1_PREFIX}_meta.json") as _f:
    META = json.load(_f)
SEGMENTATION = META.get("segmentation")
if SEGMENTATION not in ("dynamic", "predefined"):
    raise SystemExit(f"{L1_PREFIX.name}_meta.json records segmentation {SEGMENTATION!r}: use a handoff written by this "
                     "pipeline's LVL_1")
print(f"LVL_1 handoff {L1_PREFIX.name}: {SEGMENTATION} segmentation"
      + (f" (blocks from column {META.get('predefined_block_col')!r})" if SEGMENTATION == "predefined" else ""))
# the a-priori inattent_behav run (§0b, config inattent999000) is added to SWEEP_DIRS automatically
GROUPS = ("cluster", "inattent")
family = lambda rm: re.sub(r"_\d+$", "", rm)     # cluster_2 / cluster_3 -> cluster; mixture_gmm_3 -> mixture_gmm
RULES = ["speed"]              # labelling rule(s) to show: any of "speed", "size", "irr".
                               # One rule -> plain names (cluster_3); several -> side by side
                               # (cluster_3·speed, cluster_3·size, ...). The sweep saves all three.
MIN_OVERLAP_BLOCKS = 0        # a config must flag at least this many dynamic blocks to be an overlap candidate
ALPHA_SIG = 0.05
# Set-overlap statistic for every overlap output in this report (calibration picks, top-5 lists, plateau
# figure, agreement heatmaps): "jaccard" = |A ∩ B| / |A ∪ B|, or "dice" =
# 2|A ∩ B| / (|A| + |B|) = 2J / (1 + J), the Sørensen-Dice coefficient (DSC). DSC is strictly increasing in
# J, so every pick, tie and ranking is identical under either; only the reported values and labels change.
OVERLAP_STAT = "dice"         # "jaccard" | "dice"
if OVERLAP_STAT not in ("jaccard", "dice"):
    raise SystemExit(f"OVERLAP_STAT must be 'jaccard' or 'dice', got {OVERLAP_STAT!r}")
OV_NAME, OV_SYM = {"jaccard": ("Jaccard", "J"), "dice": ("Dice (DSC)", "DSC")}[OVERLAP_STAT]

# Exponent range to SHOW, per method (inclusive); None = no bound. Rows outside are dropped from every section.
EXP_RANGE = {
    "cluster":     {"kmeans_exp_entropy_exponent": (None, None), "kmeans_exp_rt_var_exponent": (None, None)},
    "mixture_gmm": {"kmeans_exp_entropy_exponent": (None, None), "kmeans_exp_rt_var_exponent": (None, None)},
}
DEGENERATE_DSC = 0.20       # method default: configurations with DSC below this are degenerate (the "poor" band, Savio et al., 2017)
SEED = 26                   # seed of the dose-response bootstraps
B_AUC = 1000                # resamples for the dose-response AUC contrasts (of distinct solutions; of configurations)
N_JOBS = max(1, (__import__("os").cpu_count() or 2) - 2)

def load(suffix=""):
    """Pool every sweep directory into one frame.

    Config ids are grid POSITIONS and restart at 0 in every --outdir, so two sweep
    directories can hold the same `clusterNNNNNN` stem for entirely different knob
    values. Pooling them under the bare stem would let groupby("config") hand one
    config's rows another's knobs, silently. Any directory that shares a stem with
    another is therefore namespaced as "<dir>:<stem>"; directories with no clash
    (e.g. the a-priori dir) keep the plain stem so existing references still match."""
    files = []
    for f in sorted(f for d in SWEEP_DIRS for f in d.glob("*.csv")):
        base = f.stem[: len(f.stem) - len(suffix)] if suffix else f.stem
        if suffix and not f.stem.endswith(suffix):
            continue
        g = base.rstrip("0123456789")
        if g in GROUPS and base[len(g):].isdigit():
            files.append((f, base, g))
    if not files:
        return pd.DataFrame()
    _n = {}
    for _, base, _g in files:
        _n[base] = _n.get(base, 0) + 1
    _clash = {f.parent.name for f, base, _g in files if _n[base] > 1}
    if _clash:
        print(f"  config ids namespaced by directory (stem collisions): {sorted(_clash)}")
    out = [pd.read_csv(f).assign(
               group=g, config=(f"{f.parent.name}:{base}" if f.parent.name in _clash else base))
           for f, base, g in files]
    return pd.concat(out, ignore_index=True)

def clean_cfg(d):
    for c in [c for c in d.columns if c.startswith("cfg_")]:
        num = pd.to_numeric(d[c], errors="coerce")
        d[c] = num if num.notna().sum() == d[c].notna().sum() else d[c].where(d[c].isna(), d[c].astype(str).str.strip("'"))
    return d

# Provenance: every sweep directory must carry the current launcher's _provenance.json,
# its results must match it, and all directories must share one lvl2.py + LVL_1 handoff.
_provs = [read_sweep(d, r"cluster\d{6}", {k: k for k in ("pipeline_fingerprint", "lvl1_input", "condition")})
          for d in SWEEP_DIRS]
if len({(p["pipeline_fingerprint"], p["lvl1_input"], p["condition"]) for p in _provs}) != 1:
    raise SystemExit(f"sweep directories disagree on lvl2.py / LVL_1 handoff / condition: {_provs}")
SWEEP_FP, SWEEP_LVL1, SWEEP_COND = (_provs[0][k] for k in ("pipeline_fingerprint", "lvl1_input", "condition"))
print(f"sweep provenance: lvl2.py {SWEEP_FP}, LVL_1 handoff {SWEEP_LVL1}, condition {SWEEP_COND}")
if L1_PREFIX.name != SWEEP_LVL1:
    raise SystemExit(f"LVL1_HANDOFF names {L1_PREFIX.name}, but the sweep ran on {SWEEP_LVL1}")
# Completeness: config ids are grid positions 0 .. N-1, and the cfg_ columns of a folder's result files (without the
# irr_mm_* metric switches the launcher adds) rebuild the configuration list whose hash is its _provenance.json "grid".
# A folder is read only when its result files (each with its _h2 and _flagged file) are that whole grid. Otherwise the
# configurations that failed or never ran are listed, with the launcher's errors/ files for the folder, and the report
# stops.
ERR_DIR = CODE_DIR / "errors"


def _cfg_value(s):
    """A cfg_ value as the launcher passed it (the result files hold repr() of each value)."""
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError):
        return s


for _d, _p in zip(SWEEP_DIRS, _provs):
    _files = {int(f.stem[-6:]): f for f in _d.glob("*.csv") if re.fullmatch(r"cluster\d{6}", f.stem)}
    _n = max(_files) + 1 if _files else 0
    _gaps = [f"cluster{i:06d}" for i in range(_n) if i not in _files]
    _rows, _incomplete = {}, []
    for _i, _f in sorted(_files.items()):
        _lack = [s for s in ("_h2", "_flagged") if not _f.with_name(f"{_f.stem}{s}.csv").is_file()]
        if _lack:
            _incomplete.append(f"{_f.stem} (no {' / '.join(_lack)} file)")
            continue
        _r = pd.read_csv(_f, nrows=1, keep_default_na=False, dtype=str)
        if _r.empty:                      # header only (the configuration flagged nothing): its knobs are in _h2
            _r = pd.read_csv(_f.with_name(f"{_f.stem}_h2.csv"), nrows=1, keep_default_na=False, dtype=str)
        if not _r.empty:
            _rows[_i] = {c[4:]: _cfg_value(_r[c].iloc[0]) for c in _r.columns
                         if c.startswith("cfg_") and not c.startswith("cfg_irr_mm_")}
    _hash_ok = bool(_files) and len(_rows) == _n and grid_fingerprint([_rows[i] for i in range(_n)]) == _p["grid"]
    if _gaps or _incomplete or not _hash_ok:
        _errs = sorted(f.name for f in ERR_DIR.glob(f"{_d.name}_*.err"))
        raise SystemExit(
            f"{_d.name} is not the launcher's whole grid: {len(_files)} result files with ids up to {_n - 1}, whose knobs "
            f"{'rebuild' if _hash_ok else 'do not rebuild'} the grid hashed in _provenance.json ({_p['grid']}). Missing "
            f"configurations: {', '.join(_gaps) or 'none below the highest id'}"
            f"{'; incomplete: ' + ', '.join(_incomplete) if _incomplete else ''}; errors/ files: {', '.join(_errs) or 'none'}. "
            "Re-run the launcher on this folder (it resumes) before reading it.")
    print(f"{_d.name}: {_n} configurations = the launcher's whole grid (hash {_p['grid']}), each with its _h2 and _flagged file")
# Level-1 RT-threshold handling of that handoff (from its _meta.json)
_l1_mode = META["lvl1_config"].get("rt_threshold_na_handling", "not recorded in its meta")
print(f"LVL_1 handoff rt_threshold_na_handling: {_l1_mode}")
if pipeline_fingerprint(CODE_DIR / "lvl2.py") != SWEEP_FP:
    print("WARNING: lvl2.py has changed since this sweep ran; the sweep describes the OLD pipeline.")
# The script the sweep ran: its byte copy _script_lvl2.py in the first sweep folder, which must have the sweep's
# fingerprint. A missing copy is written from lvl2.py, only when lvl2.py still has that fingerprint. Its LVL_2_CONFIG
# (the dict literal of NOTEBOOK CELL 28) is the base of the selection's LVL_2_CONFIG_full ([cell 16]).
SCRIPT_COPY = SWEEP_DIRS[0] / "_script_lvl2.py"
if not SCRIPT_COPY.is_file():
    if pipeline_fingerprint(CODE_DIR / "lvl2.py") != SWEEP_FP:
        raise SystemExit(f"{SCRIPT_COPY.name} is missing in {SWEEP_DIRS[0]} and lvl2.py does not have the sweep's "
                         f"fingerprint {SWEEP_FP}: the script the sweep ran is not available")
    SCRIPT_COPY.write_bytes((CODE_DIR / "lvl2.py").read_bytes())
    SCRIPT_COPY.chmod(0o444)
    print(f"script copy written: {SCRIPT_COPY}")
if pipeline_fingerprint(SCRIPT_COPY) != SWEEP_FP:
    raise SystemExit(f"{SCRIPT_COPY} has fingerprint {pipeline_fingerprint(SCRIPT_COPY)}, not the sweep's {SWEEP_FP}: "
                     "it is not the script the sweep ran")
_cfg_node = [n for n in ast.parse(SCRIPT_COPY.read_bytes()).body if isinstance(n, ast.Assign)
             and [getattr(t, "id", None) for t in n.targets] == ["LVL_2_CONFIG"]]
if len(_cfg_node) != 1:
    raise SystemExit(f"{SCRIPT_COPY.name} has {len(_cfg_node)} top-level LVL_2_CONFIG assignments, not one")
SWEEP_LVL2_CONFIG = json.loads(json.dumps(ast.literal_eval(_cfg_node[0].value)))   # JSON values: tuples -> lists
print(f"script copy: {SCRIPT_COPY} (fingerprint {pipeline_fingerprint(SCRIPT_COPY)}; LVL_2_CONFIG: "
      f"{len(SWEEP_LVL2_CONFIG)} keys)")

df, h2, flag = clean_cfg(load()), clean_cfg(load("_h2")), load("_flagged")
if df.empty:
    raise SystemExit(f"no finished configs in {SWEEP_DIRS}")
RULE = "cfg_worst_cluster_idx"
cfg_tab = df.groupby("config")[[c for c in df.columns if c.startswith("cfg_")] + ["group"]].first()
# The method whose blocks the sweep labelled (the LVL1 selection's detector, or Predefined in predefined segmentation);
# the a-priori run (§0b) uses the same one.
_ams = sorted(cfg_tab["cfg_analysis_method"].dropna().astype(str).str.strip("'\"").unique())
if len(_ams) != 1:
    raise SystemExit(f"the sweep folders mix analysis methods {_ams}; pool one method's folders only")
SWEEP_AM = _ams[0]
if SEGMENTATION == "predefined" and SWEEP_AM != "Predefined":
    raise SystemExit(f"the sweep labelled the blocks of {SWEEP_AM}, but {L1_PREFIX.name} declares predefined "
                     "segmentation (method Predefined)")
if META.get("analysis_method") not in (None, SWEEP_AM):
    raise SystemExit(f"the sweep labelled the blocks of {SWEEP_AM}, but {L1_PREFIX.name} analyses "
                     f"{META['analysis_method']!r} (its _meta.json)")
print(f"analysis method (design \"dynamic\"): {SWEEP_AM}")

# Interpreter and package versions of this run.
import sys as _vsys, platform as _vplatform
from importlib import metadata as _vmeta


def _pkg_version(p):
    try:
        return _vmeta.version(p)
    except _vmeta.PackageNotFoundError:
        return "not installed"


print(f"interpreter: {_vsys.executable} (Python {_vplatform.python_version()})")
print("packages: " + ", ".join(f"{p} {_pkg_version(p)}" for p in (
    "numpy", "pandas", "scipy", "scikit-learn", "statsmodels", "hmmlearn", "ruptures", "joblib", "matplotlib")))


*[cell 2]*

### 0b. A-priori `inattent_behav` configuration (top gold standard)

Written by `launch_LV2_sweep_random.py --stage apriori` into the sweep folder's `apriori/`; read here, with a check that
it comes from the sweep's `lvl2.py` and LVL_1 handoff. Its record (`inattent999000_heuristic.json`) also holds the
segmentation, the designs of the sweep's runs and their labels, which the rest of this report uses.


In [ ]:
# [cell 3]
import ast, sys

# The a-priori gold standard is the behavioural heuristic of LVL_2_CONFIG ([cell 28] of the LVL2 notebook, built into
# lvl2.py), run once by launch_LV2_sweep_random.py --stage apriori on the sweep's handoff and condition. This report
# reads it; it must come from the sweep's lvl2.py and handoff.
APRIORI_CONFIG = "inattent999000"
APRIORI_DIR = CACHE_DIR / "apriori"
APRIORI_COND = "all"
if SWEEP_COND != APRIORI_COND:
    raise SystemExit(f"the sweep ran on condition {SWEEP_COND}; the a-priori standard is {APRIORI_COND}")
out = APRIORI_DIR / f"{APRIORI_CONFIG}.csv"
if not out.exists():
    raise SystemExit(f"{out} is missing: run launch_LV2_sweep_random.py --stage apriori on this sweep first")
_o = pd.read_csv(out, nrows=1)
_got = {k: (str(_o[k].iloc[0]) if k in _o.columns and len(_o) else None) for k in ("lvl1_input", "pipeline_fingerprint")}
if _got != {"lvl1_input": SWEEP_LVL1, "pipeline_fingerprint": SWEEP_FP}:
    raise SystemExit(f"{out} was made from {_got}, not from this sweep's (lvl1_input={SWEEP_LVL1}, "
                     f"pipeline_fingerprint={SWEEP_FP}); re-run the apriori stage")
if APRIORI_DIR not in SWEEP_DIRS:
    SWEEP_DIRS.append(APRIORI_DIR)
df, h2, flag = clean_cfg(load()), clean_cfg(load("_h2")), load("_flagged")   # reload with the a-priori run
_AP_RECORD = json.load(open(APRIORI_DIR / f"{APRIORI_CONFIG}_heuristic.json"))
APRIORI_HEURISTIC = _AP_RECORD["heuristic"]
# The designs of the sweep's lvl2.py runs and their labels, as the a-priori run recorded them (LVL2 [cell 29]; never
# recomputed here): "dynamic" = the blocks of the analysis method (label "Dynamic", or "Predefined" in predefined
# segmentation), "control" in dynamic segmentation only, "session" only when a participant has 2 or more sessions,
# "participant" always.
_ap_missing = [k for k in ("segmentation", "designs", "design_labels") if k not in _AP_RECORD]
if _ap_missing:
    raise SystemExit(f"{APRIORI_CONFIG}_heuristic.json records no {_ap_missing}: it was not written by this pipeline's "
                     "lvl2.py; re-run the apriori stage")
if _AP_RECORD["segmentation"] != SEGMENTATION:
    raise SystemExit(f"{APRIORI_CONFIG}_heuristic.json records {_AP_RECORD['segmentation']} segmentation, but the LVL_1 "
                     f"handoff {SEGMENTATION}")
DESIGNS = list(_AP_RECORD["designs"])
DES_LAB = {d: _AP_RECORD["design_labels"][d] for d in DESIGNS}
print(f"a-priori config: {out}")
print(f"designs of the sweep's runs: {', '.join(f'{d} ({DES_LAB[d]})' for d in DESIGNS)}")
print("A-priori gold standard: behavioural heuristic (inattent_behav)")
print("-" * 62)
for k, v in APRIORI_HEURISTIC.items():
    print(f"  {k:<38s} {str(v['value']):<14s} [{v['source']}]")

In [ ]:
# [cell 4]
RULE = "cfg_worst_cluster_idx"

def _add_rule(d):
    """Labelling rule per row: the Rule column (multi-rule sweeps save every rule) or, for
    single-rule sweep files, the config's worst_cluster_idx. NaN for inattent_behav."""
    if not len(d):
        return d
    fam = d["Removal_Method"].map(family).isin(["cluster", "mixture_gmm"])
    if "Rule" in d.columns:
        r = d["Rule"]
    elif RULE in d.columns:
        r = d[RULE]
    elif RULE in cfg_tab.columns:                # _flagged files carry no cfg_ columns
        r = d["config"].map(cfg_tab[RULE])
    else:
        r = pd.Series(np.nan, index=d.index)
    return d.assign(Rule=r.where(fam))

df, h2, flag = (_add_rule(d) for d in (df, h2, flag))
df, h2, flag = (d[d["Rule"].isna() | d["Rule"].isin(RULES)] if len(d) else d for d in (df, h2, flag))

def _in_range(d):                            # per-method exponent range filter
    ok = pd.Series(True, index=d.index)
    for rm, knobs in EXP_RANGE.items():
        for knob, (lo, hi) in knobs.items():
            col = f"cfg_{knob}"
            if col not in d.columns:
                continue
            row = d["Removal_Method"].map(family) == rm
            if lo is not None:
                ok &= ~row | (d[col] >= lo)
            if hi is not None:
                ok &= ~row | (d[col] <= hi)
    return ok

df, h2 = df[_in_range(df)], h2[_in_range(h2)]
print(f"exponent range filter: {EXP_RANGE}")

def method_label(d):
    """<Removal_Method>·<Rule> for cluster / GMM arms, the bare name otherwise."""
    return np.where(d["Rule"].notna(), d["Removal_Method"] + "·" + d["Rule"].astype(str), d["Removal_Method"])

df["method"] = method_label(df)
if len(h2):
    h2["method"] = method_label(h2)
flag["method"] = method_label(flag)
# Fits a configuration's lvl2.py could not make (Step 5; e.g. a GMM with k components needs at least 5 k units with every
# feature in a design's unit matrix): its _h2.csv row with the reason in fit_failure. {(config, arm, design): reason}.
# They have no flags and are never counted as flagging 0 units ([cell 7], [cell 8]).
NOT_FITTED = ({(c, a, d): w for c, a, d, w in h2.loc[h2["fit_failure"].notna(),
                                                     ["config", "Removal_Method", "Design", "fit_failure"]].values}
              if "fit_failure" in h2.columns else {})
if NOT_FITTED:
    print(f"not fitted: {len(NOT_FITTED)} configuration x arm x design fit(s) ([cell 8], dsc_not_fitted)")
_pairs = set(zip(df.config, df.method))
flag = flag[[(c, m) in _pairs for c, m in zip(flag.config, flag.method)]]
# alpha change per trial removed (absolute); Pct_Removed is the descriptive share of trials removed
df["Alpha_ImprovementPerTrial"] = df["Alpha_Improvement"] / df["Trials_Removed"]
df["Alpha_ExcessPerTrial"] = df["Alpha_Excess"] / df["Trials_Removed"]
VALS = ["Alpha_Improvement", "Alpha_ImprovementPerTrial", "Alpha_Excess", "Alpha_ExcessPerTrial", "Pct_Removed",
        "Trials_Removed"]
COMPARATORS = DESIGNS[1:]          # DESIGNS: the designs of the sweep's runs ([cell 3])

CFG_COLS = [c for c in df.columns if c.startswith("cfg_")]
cfg_tab = df.groupby("config")[CFG_COLS + ["group"]].first()
agg = (df.groupby(["config", "method", "Removal_Method", "Design"])[VALS].mean()
         .reset_index().merge(cfg_tab, left_on="config", right_index=True))
METHODS = [m for m in ["inattent_behav"] + sorted(set(agg.method) - {"inattent_behav"}) if m in set(agg.method)]

flag["key"] = (flag["id"].astype(str) + "_" + flag["session"].astype(float).astype(int).astype(str)
               + "_" + flag["block"].astype(float).astype(int).astype(str))
SETS = flag.groupby(["config", "Design", "method"])["key"].agg(set).to_dict()
get = lambda cfg, m, design="dynamic": SETS.get((cfg, design, m), set())   # m = method label
jac = lambda a, b: (((2 * len(a & b) / (len(a) + len(b))) if OVERLAP_STAT == "dice"   # OVERLAP_STAT, cell 1
                     else len(a & b) / len(a | b)) if (a or b) else np.nan)
short = lambda m: m.split("·")[0] if ("·" in m and len(RULES) == 1) else m

print(f"{df.config.nunique():,} configs  |  rules shown: {RULES}  |  methods: {METHODS}")
print(df.groupby("group")["config"].nunique().to_string())


*[cell 5]*

## 1. Calibration to the a-priori standard

Cluster and GMM each pick, independently, the configuration whose flagged `dynamic` blocks (the analysis method's) overlap the a-priori
`inattent_behav` flags most (Jaccard or Dice per `OVERLAP_STAT`; candidates must flag >= `MIN_OVERLAP_BLOCKS`). The top-5 lists show whether the
pick is a clear winner or one point on a plateau.

**Picks.** Within each arm, the config with the greatest dynamic overlap (exact ties broken by **cluster stability**, the notebook's bootstrap stability: median adjusted Rand index between refits on resampled blocks and the full-data partition). Between the arms of a family (cluster_2 vs cluster_3; mixture_gmm_2 vs mixture_gmm_3), the pick with the **higher stability**. Stability uses neither α nor the gold standard, so it is not circular.

In [ ]:
# [cell 6]
# ---------------- 1a. Tie-break by cluster stability --------------------------------------------------
# The pick is the config with the greatest overlap (OVERLAP_STAT) with the a-priori standard. When several configs tie
# exactly, or two arms of one family (cluster_2 / cluster_3; mixture_gmm_2 / mixture_gmm_3) tie, the tie is broken by
# CLUSTER STABILITY: refit on blocks resampled with replacement and take the median adjusted Rand index (ARI) between the
# refit and the full-data partition. The refits are launch_LV2_sweep_random.py --stage stability's (stability/: one JSON
# of summaries and one NPZ of the ARIs per candidate config); it uses neither the outcome (alpha) nor the gold
# standard, so it is not circular. This report reads them and checks that its candidates are the stage's.
STAB_DIR = CACHE_DIR / "stability"
# The refits per candidate: the launcher's, stability_refits of the sweep's lvl2.py (its script copy, [cell 1]), which
# the stability files record (meta reps).
STAB_REPS = int(SWEEP_LVL2_CONFIG["stability_refits"])
_cand_file = STAB_DIR / "_candidates.json"
if not _cand_file.exists():
    raise SystemExit(f"{_cand_file} is missing: run launch_LV2_sweep_random.py --stage stability on this sweep first")
STAB_CANDIDATES = {c: set(a) for c, a in json.loads(_cand_file.read_text()).items()}


def _stability(requests):
    """requests: {config: set(arms)} -> {(config, arm): summary dict}, from the stability stage's files (same lvl2.py,
    handoff, condition and number of refits as this sweep)."""
    meta = dict(pipeline_fingerprint=SWEEP_FP, lvl1_input=SWEEP_LVL1, condition=SWEEP_COND, reps=STAB_REPS)
    out = {}
    for cfg, arms in requests.items():
        path = STAB_DIR / f"{cfg}.json"
        cached = json.loads(path.read_text()) if path.exists() else None
        if cached and {**cached.get("meta", {}), "reps": STAB_REPS} == meta and cached["meta"].get("reps") != STAB_REPS:
            raise SystemExit(f"stability of {cfg} was refitted {cached['meta'].get('reps')} times, not the sweep's "
                             f"stability_refits = {STAB_REPS}: run launch_LV2_sweep_random.py --stage stability on it")
        if not (cached and cached.get("meta") == meta and set(arms) <= set(cached["arms"])):
            raise SystemExit(f"stability of {cfg} {sorted(arms)} is missing or was made from another lvl2.py / handoff: "
                             "run launch_LV2_sweep_random.py --stage stability")
        out.update({(cfg, a): cached["arms"][a] for a in arms})
    return out


def _ties(scores):
    """Keys whose score equals the maximum exactly (Jaccard and Dice are ratios of counts)."""
    top = max(scores.values())
    return [c for c, v in scores.items() if np.isclose(v, top, rtol=0, atol=1e-12)]


In [ ]:
# [cell 7]
SWEPT = [c for c in CFG_COLS if cfg_tab[c].nunique(dropna=True) > 1]   # knobs that actually vary
CM = [m for m in METHODS if family(m.split("·")[0]) in ("cluster", "mixture_gmm")]
dyn = agg[agg.Design == "dynamic"]

def best_config(method, col):
    d = dyn[(dyn.method == method)].dropna(subset=[col])
    return d.sort_values(col, ascending=False).iloc[0].config if len(d) else None

GOLD, rows = {}, []

# top gold standard: cluster / GMM calibrated to the A-PRIORI inattent definition
if APRIORI_CONFIG:
    _ref = get(APRIORI_CONFIG, "inattent_behav")
    _W = {"inattent_behav": APRIORI_CONFIG}
    _arm = lambda m: m.split("·")[0]
    SCORES = {}
    for m in CM:
        # a configuration whose lvl2.py could not fit the arm on the analysed blocks (NOT_FITTED, [cell 4]) has no score
        sc = {c: jac(_ref, get(c, m)) for c in sorted(set(agg.loc[agg.method == m, "config"]))
              if len(get(c, m)) >= MIN_OVERLAP_BLOCKS and (c, _arm(m), "dynamic") not in NOT_FITTED}
        if sc:
            SCORES[m] = sc
    FAMS = {}
    for m in SCORES:
        FAMS.setdefault(family(_arm(m)), []).append(m)
    # cluster stability of every arm's top-overlap config (all of them when several tie exactly)
    _req = {}
    for m, sc in SCORES.items():
        for cf in _ties(sc):
            _req.setdefault(cf, set()).add(_arm(m))
    STAB = _stability(_req)
    if {c_: set(a_) for c_, a_ in _req.items()} != STAB_CANDIDATES:
        raise SystemExit(f"the tie candidates of this report {_req} differ from the stability stage's {STAB_CANDIDATES}")
    _s = lambda c, m: STAB[(c, _arm(m))]
    TIES = []
    for m, sc in SCORES.items():
        t = _ties(sc)
        _W[m] = t[0] if len(t) == 1 else max(t, key=lambda c: _s(c, m)["median_ari"])
        TIES += [dict(level="config", method=short(m), config=cf, **{OV_SYM: sc[cf]}, **{k: _s(cf, m)[k] for k in
                      ("median_ari", "q25", "q75", "mean_ari", "sd_ari", "q025", "q975")}, picked=(cf == _W[m])) for cf in t]
    GOLD["a-priori inattent overlap"] = _W
    # family pick (cluster_2 vs cluster_3; mixture_gmm_2 vs mixture_gmm_3): each arm's pick is
    # made by dynamic overlap above; between the arms of a family, the more STABLE pick wins. The LVL_2 run labels every
    # design with the family's pick, so an arm whose calibrated configuration could not be fitted on some design's units
    # (NOT_FITTED, [cell 4]) is not a candidate.
    _unfit = {m: [(d, NOT_FITTED[(_W[m], _arm(m), d)]) for d in DESIGNS if (_W[m], _arm(m), d) in NOT_FITTED]
              for m in SCORES}
    for m, u in _unfit.items():
        if u:
            print(f"{short(m)}: not a family pick: its calibrated configuration {_W[m]} was not fitted on the "
                  + "; ".join(f"{d} units ({why})" for d, why in u))
    FAMILY_PICK = {}
    for fam, ms in FAMS.items():
        if all(_unfit[m] for m in ms):
            raise SystemExit(f"no {fam} arm's calibrated configuration was fitted on every design's units "
                             f"({'; '.join(f'{short(m)}: {_unfit[m]}' for m in ms)}). A GMM with k components needs at "
                             "least 5 k units with every feature in each design's unit matrix: the data hold too few "
                             "units for this family")
        win = max((m for m in ms if not _unfit[m]), key=lambda m: _s(_W[m], m)["median_ari"])
        FAMILY_PICK[fam] = dict(method=short(win), config=_W[win], **{OV_SYM: SCORES[win][_W[win]]},
                                median_ari=_s(_W[win], win)["median_ari"], q25=_s(_W[win], win)["q25"],
                                q75=_s(_W[win], win)["q75"], rule="higher stability (median ARI)")
        TIES += [dict(level="family", method=short(m), config=_W[m], **{OV_SYM: SCORES[m][_W[m]]},
                      **{k: _s(_W[m], m)[k] for k in ("median_ari", "q25", "q75", "mean_ari", "sd_ari", "q025", "q975")}, picked=(m == win))
                 for m in ms]
    if TIES:
        print("cluster stability (bootstrap ARI: median, IQR q25-q75, mean, SD, 95% range over resamples). level=config: each arm's "
              "top-overlap config(s), exact overlap ties decided by stability; level=family: the arms' picks, "
              "the more stable one (higher median ARI) is the family pick:")
        _tt = pd.DataFrame(TIES)
        _tt["kmeans_exp_entropy_exponent"] = cfg_tab.loc[_tt.config, "cfg_kmeans_exp_entropy_exponent"].to_numpy()
        _tt["kmeans_exp_rt_var_exponent"] = cfg_tab.loc[_tt.config, "cfg_kmeans_exp_rt_var_exponent"].to_numpy()
        display(_tt.round(4))
    print("family picks:")
    _fp = pd.DataFrame(FAMILY_PICK).T
    _fp.insert(2, "kmeans_exp_entropy_exponent", cfg_tab.loc[_fp.config, "cfg_kmeans_exp_entropy_exponent"].to_numpy())
    _fp.insert(3, "kmeans_exp_rt_var_exponent", cfg_tab.loc[_fp.config, "cfg_kmeans_exp_rt_var_exponent"].to_numpy())
    display(_fp)
# how close the runner-up configs are (is the pick a clear winner or one of a plateau?)
if APRIORI_CONFIG:
    for m in CM:
        rm = m.split("·")[0]
        sc = pd.Series({c: jac(_ref, get(c, m)) for c in sorted(set(agg.loc[agg.method == m, "config"]))
                        if len(get(c, m)) >= MIN_OVERLAP_BLOCKS and (c, rm, "dynamic") not in NOT_FITTED},
                       dtype=float).sort_values(ascending=False).head(5)
        print(f"{short(m)}: top 5 overlap with the a-priori standard")
        display(pd.DataFrame({OV_NAME: sc, "kmeans_exp_entropy_exponent": cfg_tab.loc[sc.index, "cfg_kmeans_exp_entropy_exponent"],
                              "kmeans_exp_rt_var_exponent": cfg_tab.loc[sc.index, "cfg_kmeans_exp_rt_var_exponent"]}).round(3))

# The control columns only when "control" is a design of the sweep's runs (dynamic segmentation); the suffix of the
# analysis method's columns follows its label (dyn, or pre in predefined segmentation).
_CTL = "control" in DESIGNS
_DS = f"({DES_LAB['dynamic'][:3].lower()})"
for name, W in GOLD.items():
    for m, cfg in W.items():
        w = agg[(agg.config == cfg) & (agg.method == m)].set_index("Design")
        hb_d = h2[(h2.config == cfg) & (h2.method == m) & (h2.Design == "dynamic")]
        hb_c = h2[(h2.config == cfg) & (h2.method == m) & (h2.Design == "control")]
        anchor = W.get("inattent_behav")
        _d = lambda col: w.loc["dynamic", col] if "dynamic" in w.index else np.nan
        _c = lambda col: w.loc["control", col] if "control" in w.index else np.nan
        rows.append({"gold standard": name, "method": short(m), "config": cfg,
                     # irr_removal_methods is constant within a sweep group, so it is
                     # dropped here rather than repeated on every row.
                     **{k.replace("cfg_", ""): cfg_tab.loc[cfg, k] for k in SWEPT
                        if k != "cfg_irr_removal_methods"
                        and cfg_tab.loc[cfg, k] == cfg_tab.loc[cfg, k]},
                     f"% blocks flagged {_DS}": hb_d.pct_blocks_flagged.iloc[0] if len(hb_d) else np.nan,
                     **({"% blocks flagged (ctl)": hb_c.pct_blocks_flagged.iloc[0] if len(hb_c) else np.nan}
                        if _CTL else {}),
                     f"% trials {_DS}": _d("Pct_Removed"),
                     **({"% trials (ctl)": _c("Pct_Removed")} if _CTL else {}),
                     f"{OV_SYM} w/ inattent": jac(get(anchor, "inattent_behav"), get(cfg, m)) if anchor else np.nan,
                     f"α excess {_DS}": _d("Alpha_Excess"),
                     **({"α excess (ctl)": _c("Alpha_Excess")} if _CTL else {}),
                     f"α excess per trial ×10⁻⁶ {_DS}": 1e6 * _d("Alpha_ExcessPerTrial"),
                     **({"α excess per trial ×10⁻⁶ (ctl)": 1e6 * _c("Alpha_ExcessPerTrial")} if _CTL else {}),
                     **({"ARI mean": _ari["mean_ari"], "ARI SD": _ari["sd_ari"], "ARI median": _ari["median_ari"],
                         "ARI IQR": f"{_ari['q25']:.3f}-{_ari['q75']:.3f}"}
                        if (_ari := (STAB.get((cfg, m.split("·")[0])) if APRIORI_CONFIG else None))
                        else {"ARI mean": np.nan, "ARI SD": np.nan, "ARI median": np.nan, "ARI IQR": "-"})})

GOLD_TAB = pd.DataFrame(rows).set_index(["gold standard", "method"])
GOLD_TAB = GOLD_TAB.dropna(axis=1, how="all")
print("Swept knobs only; every other LVL_2_CONFIG value is identical across configs (printed in full below).")
display(GOLD_TAB.round(4))

In [ ]:
# [cell 8]
# ---------------- 1b. DSC with the a-priori standard across the sweep: degeneracy, the exponents, the maxima -----
# Every configuration x arm x design with flagged units (SDES, below) against the a-priori standard's flagged units of
# the same design. Tables written to LVL_OUTS as <sweep>_report_<table>.csv; the final report prints them.
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm


def smad(x):
    """Scaled MAD = 1.4826 x median absolute deviation."""
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    return 1.4826 * np.median(np.abs(x - np.median(x))) if len(x) else np.nan


def summ(x):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    if not len(x):
        return dict(n=0, median=np.nan, q1=np.nan, q3=np.nan, smad=np.nan)
    return dict(n=len(x), median=np.median(x), q1=np.percentile(x, 25), q3=np.percentile(x, 75), smad=smad(x))


def summ_str(x, nd=3):
    s = summ(x)
    return "—" if not s["n"] else f"{s['median']:.{nd}f} [{s['q1']:.{nd}f}, {s['q3']:.{nd}f}] ({s['smad']:.{nd}f})"


def ci(star, lo=2.5, hi=97.5):
    star = np.asarray(star, float); star = star[np.isfinite(star)]
    return (np.percentile(star, lo), np.percentile(star, hi)) if len(star) else (np.nan, np.nan)


def ci_str(lo, hi, nd=3):
    return "—" if not (np.isfinite(lo) and np.isfinite(hi)) else f"[{lo:.{nd}f}, {hi:.{nd}f}]"


def p_two(star, null=0.0):
    star = np.asarray(star, float); star = star[np.isfinite(star)]
    if not len(star):
        return np.nan
    n = len(star)
    return min(1.0, 2 * min((np.sum(star <= null) + 1) / (n + 1), (np.sum(star >= null) + 1) / (n + 1)))


def bh(p):
    p = np.asarray(p, float); q = np.full(len(p), np.nan); ok = np.isfinite(p)
    if ok.any():
        q[ok] = multipletests(p[ok], method="fdr_bh")[1]
    return q


TAB_STEM = LVL_OUTS / f"{SWEEP_DIRS[0].name}_report"
REPORT_TABLES = {}


def export(key, frame):
    REPORT_TABLES[key] = frame
    frame.to_csv(f"{TAB_STEM}_{key}.csv", index=False)
    display(frame)


ARMS2 = ["cluster_2", "cluster_3", "mixture_gmm_2", "mixture_gmm_3"]
dsc = lambda a, b: (2 * len(a & b) / (len(a) + len(b))) if (a or b) else np.nan
_keys = lambda g, des: (set(g["key"]) if des != "session" else {f"{int(i)}_{int(s)}" for i, s in g[["id", "session"]].values})
_ap = flag[(flag.config == APRIORI_CONFIG) & (flag.Removal_Method == "inattent_behav")]
_ap_h2 = h2[(h2.config == APRIORI_CONFIG) & (h2.Removal_Method == "inattent_behav")].set_index("Design")
# The designs the runs fit and flag (the a-priori run's _h2 rows), in DESIGNS order: the block designs and the
# per-session unit fit ("session"; "participant" when no participant has 2 or more sessions). The participant design
# removes the participants of the flagged sessions and has no flagged-unit rows of its own.
SDES = tuple(d for d in DESIGNS if d in set(_ap_h2.index))
APRIORI_SETS = {des: _keys(_ap[_ap.Design == des], des) for des in SDES}
_units = {des: int(_ap_h2.loc[des, "n_blocks"]) for des in SDES}
_fl = flag[(flag.config != APRIORI_CONFIG) & (flag.Rule == "speed")]
_cfgs = sorted(c for c in cfg_tab.index if c != APRIORI_CONFIG and cfg_tab.loc[c, "group"] == "cluster")
_ex = cfg_tab[["cfg_kmeans_exp_entropy_exponent", "cfg_kmeans_exp_rt_var_exponent"]]


def chance_dsc(g, des):
    """Expected DSC if the arm's n flagged units were drawn at random from the design's N units: 2nG / (N(n + G))."""
    G_ = len(APRIORI_SETS[des]); n_ = g.n_flagged.to_numpy(float)
    return 2 * n_ * G_ / (_units[des] * (n_ + G_))


_rows, _nf_rows = [], {}
_byc = {k: g for k, g in _fl.groupby(["config", "Removal_Method", "Design"])}
for c in _cfgs:
    for arm in ARMS2:
        for des in SDES:
            if (c, arm, des) in NOT_FITTED:                  # no fit, so no DSC: listed below, never counted as 0 flagged
                _nf_rows.setdefault((arm, des, NOT_FITTED[(c, arm, des)]), []).append(c)
                continue
            s = _keys(_byc.get((c, arm, des), _fl.iloc[:0]), des)
            _rows.append(dict(config=c, arm=arm, design=des, n_flagged=len(s), DSC=dsc(s, APRIORI_SETS[des]),
                              exp_entropy=float(_ex.loc[c].iloc[0]), exp_rt_var=float(_ex.loc[c].iloc[1])))
SWEEP_DSC = pd.DataFrame(_rows)
# The fits lvl2.py could not make ([cell 4]): left out of every DSC table below, with the reason
export("dsc_not_fitted", pd.DataFrame([{"Arm": a, "Design": d, "Configs not fitted": len(cs), "Reason": why}
                                       for (a, d, why), cs in _nf_rows.items()],
                                      columns=["Arm", "Design", "Configs not fitted", "Reason"]))
PICK_CFG = {m.split("·")[0]: c for m, c in GOLD["a-priori inattent overlap"].items() if m != "inattent_behav"}
rows = []
for (arm, des), g in SWEEP_DSC.groupby(["arm", "design"], sort=False):
    deg = g.DSC < DEGENERATE_DSC
    nd_ = summ(g.DSC[~deg])
    rows.append({"Arm": arm, "Design": des, "Configs": len(g),
                 "No overlap (DSC = 0) n (%)": f"{int((g.DSC == 0).sum())} ({100 * (g.DSC == 0).mean():.1f})",
                 "Degenerate (DSC < .20) n (%)": f"{int(deg.sum())} ({100 * deg.mean():.1f})",
                 "Flagging ≥ 99% of units": int((g.n_flagged >= 0.99 * _units[des]).sum()),
                 "DSC, all: Mdn [Q1, Q3] (sMAD)": summ_str(g.DSC),
                 "Non-degenerate n": nd_["n"], "DSC, non-degenerate: Mdn [Q1, Q3] (sMAD)": summ_str(g.DSC[~deg]),
                 "Below chance-expected DSC n": int((g.DSC < chance_dsc(g, des)).sum()),
                 "Pick DSC": g.loc[g.config == PICK_CFG[arm], "DSC"].iloc[0] if (g.config == PICK_CFG[arm]).any()
                 else np.nan})
for des, g in SWEEP_DSC.groupby("design", sort=False):
    deg = g.DSC < DEGENERATE_DSC
    rows.append({"Arm": "all arms", "Design": des, "Configs": len(g),
                 "No overlap (DSC = 0) n (%)": f"{int((g.DSC == 0).sum())} ({100 * (g.DSC == 0).mean():.1f})",
                 "Degenerate (DSC < .20) n (%)": f"{int(deg.sum())} ({100 * deg.mean():.1f})",
                 "Flagging ≥ 99% of units": int((g.n_flagged >= 0.99 * _units[des]).sum()),
                 "DSC, all: Mdn [Q1, Q3] (sMAD)": summ_str(g.DSC), "Non-degenerate n": int((~deg).sum()),
                 "DSC, non-degenerate: Mdn [Q1, Q3] (sMAD)": summ_str(g.DSC[~deg]),
                 "Below chance-expected DSC n": int((g.DSC < chance_dsc(g, des)).sum()), "Pick DSC": np.nan})
export("dsc_summary", pd.DataFrame(rows))
export("dsc_units", pd.DataFrame([{"Design": des, "Units": _units[des], "A-priori flagged": len(APRIORI_SETS[des])}
                                  for des in SDES]))


def exp_ss(g):
    """SS shares (Type II, additive model) of the two exponents in DSC; remainder = interaction / residual. NaN when the
    set cannot separate the exponents: a single level of one, or no residual degrees of freedom / a rank-deficient design
    (a small set)."""
    if g.exp_entropy.nunique() < 2 or g.exp_rt_var.nunique() < 2:
        return (np.nan,) * 3
    tot = ((g.DSC - g.DSC.mean()) ** 2).sum()
    fit = smf.ols("DSC ~ C(exp_entropy) + C(exp_rt_var)", data=g).fit()
    X = fit.model.exog
    if fit.df_resid < 1 or np.linalg.matrix_rank(X) < X.shape[1]:
        return (np.nan,) * 3
    a = anova_lm(fit, typ=2)
    return (a.loc["C(exp_entropy)", "sum_sq"] / tot, a.loc["C(exp_rt_var)", "sum_sq"] / tot,
            a.loc["Residual", "sum_sq"] / tot)


rows = []
for (arm, des), g in SWEEP_DSC.groupby(["arm", "design"], sort=False):
    for lab, gg in (("all configurations", g), ("non-degenerate (DSC ≥ .20)", g[g.DSC >= DEGENERATE_DSC])):
        e, v, r = exp_ss(gg)
        rows.append({"Arm": arm, "Design": des, "Set": lab, "Configs": len(gg), "SS share entropy exponent": e,
                     "SS share RT-variance exponent": v, "SS share interaction / residual": r})
export("dsc_variance", pd.DataFrame(rows))
rows = []
for arm, cfg in PICK_CFG.items():
    g = SWEEP_DSC[(SWEEP_DSC.arm == arm) & (SWEEP_DSC.config == cfg)].set_index("design")
    rows.append({"Arm": arm, "Config": cfg, "Entropy exponent": float(_ex.loc[cfg].iloc[0]),
                 "RT-variance exponent": float(_ex.loc[cfg].iloc[1]),
                 **{f"{d} DSC": g.loc[d, "DSC"] if d in g.index else np.nan for d in SDES},
                 **{f"{d} n flagged": int(g.loc[d, "n_flagged"]) if d in g.index else np.nan for d in SDES}})
export("dsc_picks", pd.DataFrame(rows))
rows = []
for (arm, des), g in SWEEP_DSC.groupby(["arm", "design"], sort=False):
    top = g[np.isclose(g.DSC, g.DSC.max(), rtol=0, atol=1e-12)]
    rows.append({"Arm": arm, "Design": des, "Max DSC": g.DSC.max(), "Configs at the maximum": len(top),
                 "Config(s)": ", ".join(top.config), "Entropy exponent(s)": ", ".join(f"{v:g}" for v in top.exp_entropy),
                 "RT-variance exponent(s)": ", ".join(f"{v:g}" for v in top.exp_rt_var),
                 "Calibrated config at the maximum": bool(PICK_CFG[arm] in set(top.config))})
export("dsc_maxima", pd.DataFrame(rows))
# DSC in qualitative bands (Savio et al., 2017: < .20 poor, .20-.40 fair, .40-.60 moderate, .60-.80 good, > .80
# excellent), per arm x design and over all arms: configurations per band and the % of the design's units they flag.
DSC_BANDS = [("poor", 0.0, 0.2), ("fair", 0.2, 0.4), ("moderate", 0.4, 0.6), ("good", 0.6, 0.8), ("excellent", 0.8, 1.0)]


def _band_rows(arm, des, g):
    pct = 100 * g.n_flagged.to_numpy(float) / _units[des]
    out = []
    for lab, lo, hi in DSC_BANDS:
        m = ((g.DSC >= lo) & (g.DSC < hi) if lab in ("poor", "fair", "moderate") else
             (g.DSC >= lo) & (g.DSC <= hi) if lab == "good" else (g.DSC > lo) & (g.DSC <= hi)).to_numpy()
        out.append({"Arm": arm, "Design": des, "Band": lab,
                    "DSC range": {"poor": "< .20", "fair": ".20-.40", "moderate": ".40-.60", "good": ".60-.80",
                                  "excellent": "> .80"}[lab],
                    "Configs": int(m.sum()), "Configs (%)": 100 * m.mean(),
                    "Units flagged (%): min": pct[m].min() if m.any() else np.nan,
                    "Units flagged (%): median": np.median(pct[m]) if m.any() else np.nan,
                    "Units flagged (%): max": pct[m].max() if m.any() else np.nan})
    return out


rows = [r for (arm, des), g in SWEEP_DSC.groupby(["arm", "design"], sort=False) for r in _band_rows(arm, des, g)]
rows += [r for des, g in SWEEP_DSC.groupby("design", sort=False) for r in _band_rows("all arms", des, g)]
export("dsc_bands", pd.DataFrame(rows))
SWEEP_DSC.to_csv(f"{TAB_STEM}_dsc_configs.csv", index=False)


In [ ]:
# [cell 9]
# ---------------- 1c. Cluster stability of the candidates: the 5,000 refits of the stability stage ------------------
# One row per candidate (config, arm): the arm picks, every exact DSC tie and the carried solutions. ARI between each
# refit (blocks resampled with replacement) and the full-data partition on the resampled blocks.
def ari_row(label, status, a, n):
    s = summ(a); a = a[np.isfinite(a)]
    return {"Solution": label, "Status": status, "Blocks": n, "Refits": len(a), "Median": s["median"],
            "Q1": s["q1"], "Q3": s["q3"], "sMAD": s["smad"], "Mean": a.mean(), "SD": a.std(ddof=1),
            "2.5%": np.percentile(a, 2.5), "97.5%": np.percentile(a, 97.5), "Refits ARI < .50 (%)": 100 * (a < .5).mean()}


FAM_LABEL = {"cluster": "k-means", "mixture_gmm": "GMM"}
_carried = {(fp["config"], fp["method"].split("·")[0]): fam for fam, fp in FAMILY_PICK.items()}
_arm_ties = {m.split("·")[0]: _ties(sc) for m, sc in SCORES.items()}
_top = {m.split("·")[0]: max(sc.values()) for m, sc in SCORES.items()}


def status(cfg, arm):
    """Why the candidate was refitted: a carried solution, an exact DSC tie within the arm, or the other arm of a family
    (each arm's pick is refitted; the family carries the more stable one)."""
    fam = family(arm)
    if (cfg, arm) in _carried:
        return f"{FAM_LABEL[fam]} ({arm.rsplit('_', 1)[1]}), carried"
    if len(_arm_ties[arm]) > 1:
        return f"{arm} arm pick (DSC tie)" if PICK_CFG[arm] == cfg else f"{arm} tied config (DSC tie)"
    _win = next(a for (c, a), f in _carried.items() if f == fam)
    return f"{arm} arm pick (family tie)" if np.isclose(_top[arm], _top[_win], rtol=0, atol=1e-12) else f"{arm} arm pick"


_order = lambda ca: (0 if ca in _carried else 1, list(FAM_LABEL).index(family(ca[1])), ca[1], ca[0])
rows = []
for cfg, arm in sorted(((c, a) for c, arms in STAB_CANDIDATES.items() for a in arms), key=_order):
    raw = np.load(STAB_DIR / f"{cfg}.npz")
    meta = json.loads((STAB_DIR / f"{cfg}.json").read_text())["arms"]
    a = raw[arm]
    if abs(np.nanmedian(a) - meta[arm]["median_ari"]) > 1e-12:
        raise SystemExit(f"stability {cfg} {arm}: the replicates do not reproduce their summary")
    rows.append(ari_row(f"{arm} ({cfg})", status(cfg, arm), a, meta[arm]["n_blocks"]))
export("stability", pd.DataFrame(rows))


*[cell 10]*

## 2. Dose–response of the α gain across the sweep

Saturating fits of the α gain beyond matched random removal against the share of trials removed, overall and per trial removed, their AUC over the common dose range, and paired bootstrap contrasts between the designs: resampling each arm's distinct solutions (configurations that flag the same units in every design), and, as a sensitivity analysis, its configurations.


In [ ]:
# [cell 11]
# ---------------- 2. Dose-response of the alpha gain across the sweep: saturating fits, AUC, paired bootstrap ---------
# Per configuration and design: the alpha gain beyond matched random removal (the sweep's Step 8, canonical alpha), mean
# over the scale x condition cells, against the % of trials removed. Fit: excess = a (1 - e^(-b v)), soft-L1 robust least squares. AUC =
# mean height of the fitted curve over the common dose range of the arm's designs; per trial = the alpha gain per trial
# removed (x 10^-6; per-trial curve from PER_FROM %). Design contrasts: B_AUC paired resamples, two-sided percentile p
# with the +1 correction, BH within each family (overall; per trial). dose_contrasts resamples the arm's distinct
# solutions (configurations that flag the same units in every design are one solution): each resample draws the
# solutions with replacement and refits every design on all configurations of the drawn solutions, so the point
# estimates are the all-configuration fits; the "Differs from every other design" column of dose_fits follows it.
# dose_contrasts_per_config resamples the configurations as independent units (sensitivity analysis).
from scipy.optimize import curve_fit
from joblib import Parallel, delayed
_cfg_files = sorted(SWEEP_DIRS[0].glob("cluster[0-9][0-9][0-9][0-9][0-9][0-9].csv"))
# DESIGNS: the designs of the sweep's runs ([cell 3])
sw2 = pd.concat([pd.read_csv(f).assign(config=f.stem) for f in _cfg_files], ignore_index=True)
sw2 = sw2[sw2.Rule == "speed"]
AGG2 = (sw2.groupby(["config", "Removal_Method", "Design"])[["Pct_Removed", "Alpha_Excess", "Trials_Removed"]].mean()
           .join(sw2.groupby(["config", "Removal_Method", "Design"]).Coverage_Post.min().rename("Coverage_Min"))
           .reset_index())
# Eligible removals: post-removal coverage >= the runs' floor (h2_min_coverage of the sweep script's LVL_2_CONFIG) in every
# scale. The others are left out of the fits, the common dose range and the two bootstraps; dose_fits counts them.
COV_FLOOR = float(SWEEP_LVL2_CONFIG["h2_min_coverage"])
_dosed = AGG2[AGG2.Pct_Removed > 0]
COV_EXCL = _dosed[~(_dosed.Coverage_Min >= COV_FLOOR)].groupby(["Removal_Method", "Design"]).size()
ALLPTS = _dosed[_dosed.Coverage_Min >= COV_FLOOR].dropna(subset=["Alpha_Excess"])
_pos = sw2[sw2.Pct_Removed > 0]
N_TRIALS = float((100 * _pos.Trials_Removed / _pos.Pct_Removed).groupby(_pos.Scale).mean().mean())
PER_FROM, PT = 1.0, 100.0 / N_TRIALS / 1e-6           # gain / % removed x PT = gain per trial removed, in 1e-6 units
KINDS = ["overall", "per-trial"]
_sat = lambda v, a, b: a * (1.0 - np.exp(-b * v))


def _fit_xy(x, y):
    try:
        p0, _ = curve_fit(_sat, x, y, p0=[max(y.max(), 1e-4) * 1.2, 0.1], maxfev=40000)
        r = y - _sat(x, *p0)
        f_scale = max(1.4826 * np.median(np.abs(r - np.median(r))), 1e-6)
        (a, b), _ = curve_fit(_sat, x, y, p0=p0, method="trf", loss="soft_l1", f_scale=f_scale, max_nfev=40000)
        return a, b
    except Exception:
        return np.nan, np.nan


def _r2(g, a, b, kind):
    if not np.isfinite(a):
        return np.nan
    if kind == "per-trial":
        g = g[g.Pct_Removed >= PER_FROM]
        y, pred = PT * g.Alpha_Excess / g.Pct_Removed, PT * _sat(g.Pct_Removed, a, b) / g.Pct_Removed
    else:
        y, pred = g.Alpha_Excess, _sat(g.Pct_Removed, a, b)
    return 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()


def _auc(a, b, kind, hi):
    if not np.isfinite(a):
        return np.nan
    lo = PER_FROM if kind == "per-trial" else 0.0
    xs = np.linspace(max(lo, 1e-6), hi, 400); ys = _sat(xs, a, b)
    return np.trapz(PT * ys / xs if kind == "per-trial" else ys, xs) / (hi - lo)


def _boot_chunk(data, draws):
    out = []
    for draw in draws:
        cnt = pd.Series(draw).value_counts(); row = {}
        for d, (cfg, x, y) in data.items():
            idx = np.repeat(np.arange(len(cfg)), pd.Series(cfg).map(cnt).fillna(0).to_numpy(int))
            row[d] = _fit_xy(x[idx], y[idx]) if len(idx) >= 6 else (np.nan, np.nan)
        out.append(row)
    return out


FITS, R2, DATA = {}, {}, {}
for m in ARMS2:
    for d in DESIGNS:
        s = ALLPTS[(ALLPTS.Removal_Method == m) & (ALLPTS.Design == d)]
        if len(s) < 6:
            continue
        FITS[(m, d)] = _fit_xy(s.Pct_Removed.to_numpy(float), s.Alpha_Excess.to_numpy(float))
        R2[(m, d)] = {k: _r2(s, *FITS[(m, d)], k) for k in KINDS}
        DATA.setdefault(m, {})[d] = (s.config.to_numpy(), s.Pct_Removed.to_numpy(float), s.Alpha_Excess.to_numpy(float))
HI = {m: min(ALLPTS[(ALLPTS.Removal_Method == m) & (ALLPTS.Design == d)].Pct_Removed.max() for d in DATA[m]) for m in DATA}
AUCS = {(m, kind): {d: _auc(*FITS[(m, d)], kind, HI[m]) for d in DATA[m]} for m in DATA for kind in KINDS}
# Resampling units. UNITS = the configurations an arm's fits use (any design). One distinct solution = the configurations
# that flag the same units in every design with flagged units (SDES, [cell 8]: the blocks of the block designs and the
# per-session units) in the sweep's _flagged.csv (the participant design removes the participants of the flagged
# sessions).
UNITS = {m: np.unique(np.concatenate([c for c, _, _ in DATA[m].values()])) for m in DATA}
_fl2 = pd.concat([pd.read_csv(f.with_name(f"{f.stem}_flagged.csv")).assign(config=f.stem) for f in _cfg_files],
                 ignore_index=True)
_fl2 = _fl2[_fl2.Rule == "speed"]
_fsets = (_fl2.assign(key=_fl2.id.astype(int).astype(str) + "_" + _fl2.session.astype(int).astype(str) + "_"
                      + _fl2.block.astype(int).astype(str))
              .groupby(["Removal_Method", "config", "Design"])["key"].agg(frozenset).to_dict())
SOLUTIONS = {}                                 # arm -> list of solutions, each the list of its configurations
for m in DATA:
    _by = {}
    for c in UNITS[m]:
        _by.setdefault(tuple(_fsets.get((m, c, d), frozenset()) for d in SDES), []).append(c)
    SOLUTIONS[m] = list(_by.values())
_sol_of = {m: {c: j for j, s in enumerate(SOLUTIONS[m]) for c in s} for m in DATA}
_spread = max((float(g.assign(sol=g.config.map(_sol_of[m])).groupby(["sol", "Design"])[["Pct_Removed", "Alpha_Excess"]]
                     .agg(lambda v: v.max() - v.min()).max().max())
               for m, g in AGG2[AGG2.Removal_Method.isin(list(DATA))].groupby("Removal_Method")), default=0.0)
if set(ARMS2) - set(DATA):
    print(f"dose-response: no fit for {sorted(set(ARMS2) - set(DATA))} (a fit needs 6 or more eligible configurations "
          f"of a design; this sweep has {len(_cfg_files)} configurations); their rows are absent from the dose_* tables")
print("distinct solutions / configurations resampled: "
      + "; ".join(f"{m} {len(SOLUTIONS[m])} / {len(UNITS[m])}" for m in DATA)
      + f". Largest difference in % removed or alpha excess between configurations of one solution: {_spread:.3g}")
_rng = np.random.default_rng(SEED)
DRAWS_CFG = {m: [_rng.choice(UNITS[m], size=len(UNITS[m]), replace=True) for _ in range(B_AUC)] for m in DATA}
_rng = np.random.default_rng(SEED)
DRAWS_SOL = {m: [np.concatenate([SOLUTIONS[m][j] for j in _rng.choice(len(SOLUTIONS[m]), size=len(SOLUTIONS[m]),
                                                                       replace=True)])
                 for _ in range(B_AUC)] for m in DATA}
_nchunk = N_JOBS * 2


def _contrasts(draws):
    """Paired bootstrap of the AUC contrasts between each arm's designs: every resample (configuration ids, with
    multiplicity) refits every design; two-sided percentile p with the +1 correction, BH within each family (overall;
    per trial) across all arms' contrasts."""
    tasks = [(m, draws[m][i::_nchunk]) for m in DATA for i in range(_nchunk)]
    res = Parallel(n_jobs=N_JOBS)(delayed(_boot_chunk)(DATA[m], dr) for m, dr in tasks)
    boots = {m: [] for m in DATA}
    for (m, _), rows_ in zip(tasks, res):
        boots[m].extend(rows_)
    pv = []
    for m in DATA:
        ds_ = list(DATA[m])
        for kind in KINDS:
            bt = {d: np.array([_auc(*r[d], kind, HI[m]) for r in boots[m]]) for d in ds_}
            for i, d1 in enumerate(ds_):
                for d2 in ds_[i + 1:]:
                    diff = bt[d1] - bt[d2]
                    pv.append(dict(Family=kind, Arm=m, Comparison=f"{d1} − {d2}", **{"AUC A": AUCS[(m, kind)][d1],
                              "AUC B": AUCS[(m, kind)][d2]}, Diff=AUCS[(m, kind)][d1] - AUCS[(m, kind)][d2],
                              **{"95% CI": ci_str(*ci(diff), 4)}, p=p_two(diff), n_boot=int(np.isfinite(diff).sum())))
    pv = pd.DataFrame(pv, columns=["Family", "Arm", "Comparison", "AUC A", "AUC B", "Diff", "95% CI", "p", "n_boot"])
    pv["q"] = np.nan
    for fam, g in pv.groupby("Family"):
        pv.loc[g.index, "q"] = bh(g.p.to_numpy())
    return pv


PV = _contrasts(DRAWS_SOL).assign(n_solutions=lambda t: t.Arm.map({m: len(s) for m, s in SOLUTIONS.items()}),
                                  n_configs=lambda t: t.Arm.map({m: len(u) for m, u in UNITS.items()}))
PV_CFG = _contrasts(DRAWS_CFG)
_star = lambda m, kind, d: "*" if (PV[(PV.Family == kind) & (PV.Arm == m) & PV.Comparison.str.contains(d)].q < ALPHA_SIG).all() else ""
rows = []
for m in DATA:
    for kind in KINDS:
        for d in DATA[m]:
            a, b = FITS[(m, d)]
            rows.append({"Arm": m, "Family": kind, "Design": d, "Configs": len(DATA[m][d][0]),
                         f"Removals excluded (coverage < {COV_FLOOR:.2f})": int(COV_EXCL.get((m, d), 0)), "a (ceiling)": a,
                         "b (rate)": b, "AUC": AUCS[(m, kind)][d], "AUC range (%)": f"{PER_FROM if kind == 'per-trial' else 0:.0f}–{HI[m]:.0f}",
                         "AUC range high (%)": HI[m], "R²": R2[(m, d)][kind], "R² < 0": R2[(m, d)][kind] < 0,
                         "Differs from every other design (q < .05)": bool(_star(m, kind, d))})
export("dose_fits", pd.DataFrame(rows, columns=[
    "Arm", "Family", "Design", "Configs", f"Removals excluded (coverage < {COV_FLOOR:.2f})", "a (ceiling)", "b (rate)", "AUC",
    "AUC range (%)", "AUC range high (%)", "R²", "R² < 0", "Differs from every other design (q < .05)"]))
export("dose_contrasts", PV)
export("dose_contrasts_per_config", PV_CFG)
rows = []
for m, cfg in PICK_CFG.items():
    _pk = ALLPTS[(ALLPTS.config == cfg) & (ALLPTS.Removal_Method == m) & (ALLPTS.Design == "dynamic")]
    if (m, "dynamic") not in FITS or not len(_pk):
        continue                                   # no dynamic fit for the arm, or the pick is not an eligible removal
    pk = _pk.iloc[0]
    a, b = FITS[(m, "dynamic")]
    rows.append({"Arm": m, "Config": cfg, "% trials removed": pk.Pct_Removed, "Alpha excess": pk.Alpha_Excess,
                 "Fitted excess at this dose": _sat(pk.Pct_Removed, a, b), "Per trial (× 10⁻⁶)": PT * pk.Alpha_Excess / pk.Pct_Removed,
                 "Fitted per trial (× 10⁻⁶)": PT * _sat(pk.Pct_Removed, a, b) / pk.Pct_Removed})
export("dose_picks", pd.DataFrame(rows, columns=["Arm", "Config", "% trials removed", "Alpha excess",
                                                   "Fitted excess at this dose", "Per trial (× 10⁻⁶)",
                                                   "Fitted per trial (× 10⁻⁶)"]))
ALLPTS.assign(per_trial_1e6=PT * ALLPTS.Alpha_Excess / ALLPTS.Pct_Removed).to_csv(f"{TAB_STEM}_dose_points.csv", index=False)


*[cell 15]*

## 3. The selection: configurations carried into the LVL2 run


In [ ]:
# [cell 16]
CARRY = list(GOLD)[0]      # the a-priori gold standard

print("=" * 100); print(f"CARRY TO THE LVL2 RUN:  {CARRY}"); print("=" * 100)
full = {}
for m, cfg in GOLD[CARRY].items():
    kv = {k.replace("cfg_", ""): cfg_tab.loc[cfg, k] for k in CFG_COLS if cfg_tab.loc[cfg, k] == cfg_tab.loc[cfg, k]}
    if m == "inattent_behav" and APRIORI_HEURISTIC:      # the heuristic itself (LVL_2_CONFIG, cell 3)
        kv.update({k: v["value"] for k, v in APRIORI_HEURISTIC.items()})
    full[short(m)] = pd.Series(kv)
    w = agg[(agg.config == cfg) & (agg.method == m)].set_index("Design")
    print(f"\n# {short(m)}   ({cfg};  {w.loc['dynamic', 'Pct_Removed']:.1f}% of trials removed, "
          f"α excess {w.loc['dynamic', 'Alpha_Excess']:.4f}, per trial {1e6 * w.loc['dynamic', 'Alpha_ExcessPerTrial']:.2f} x 10^-6)")
    for k, v in kv.items():
        print(f"    {k!r}: {v!r},")
print("\nAll LVL_2_CONFIG values behind the carried winners:")
display(pd.DataFrame(full).fillna("—"))

# The selection the LVL2 run must use: each family's carried arm with its configuration and exponents. The LVL2
# notebook's input check compares LVL_2_CONFIG with this file. LVL_2_CONFIG_full = every key of the sweep script's
# LVL_2_CONFIG (the script copy, [cell 1]) with each family's values; script_copy = the copy's path.
_fam_exps = {"cluster": ("kmeans_exp_entropy_exponent", "kmeans_exp_rt_var_exponent"),
             "mixture_gmm": ("mixture_gmm_exp_entropy_exponent", "mixture_gmm_exp_rt_var_exponent")}
_am = cfg_tab["cfg_analysis_method"].dropna().astype(str).str.strip("'\"").unique()
if len(_am) != 1:
    raise SystemExit(f"the sweep's configurations name {len(_am)} analysis methods: {list(_am)}")
SELECTION = {"sweep": SWEEP_DIRS[0].name, "pipeline_fingerprint": SWEEP_FP, "lvl1_input": SWEEP_LVL1,
             "condition": SWEEP_COND, "analysis_method": _am[0], "a_priori_config": APRIORI_CONFIG, "families": {}}
for fam, fp in FAMILY_PICK.items():
    arm = fp["method"].split("·")[0]; cfg = fp["config"]
    e_ent, e_var = (float(cfg_tab.loc[cfg, "cfg_kmeans_exp_entropy_exponent"]),
                    float(cfg_tab.loc[cfg, "cfg_kmeans_exp_rt_var_exponent"]))
    SELECTION["families"][fam] = {"arm": arm, "k": int(arm.rsplit("_", 1)[1]), "config": cfg, OV_SYM: fp[OV_SYM],
                                  "median_ari": fp["median_ari"],
                                  "LVL_2_CONFIG": {_fam_exps[fam][0]: e_ent, _fam_exps[fam][1]: e_var,
                                                   ("cluster_k_list" if fam == "cluster" else "gmm_k_list"):
                                                       [int(arm.rsplit("_", 1)[1])]}}
SELECTION["script_copy"] = str(SCRIPT_COPY)
SELECTION["LVL_2_CONFIG_full"] = {**SWEEP_LVL2_CONFIG, **{k: v for f in SELECTION["families"].values()
                                                         for k, v in f["LVL_2_CONFIG"].items()}}
with open(LVL_OUTS / f"{SWEEP_DIRS[0].name}_selection.json", "w") as _f:
    json.dump(SELECTION, _f, indent=2)
print(f"\nselection -> {LVL_OUTS / (SWEEP_DIRS[0].name + '_selection.json')}")
print(json.dumps(SELECTION["families"], indent=1))


*[cell 17]*

## 4. Figures

Drawn from the tables above (nothing is written unless a `*_SAVE` file name is set): **4a** the overlap with the a-priori
standard across all configurations, per arm; **4b** the profiles of the calibrated configurations; **4c** the overlap
against each exponent; **4d** the removal outcome by design at the calibrated configurations; **4e** the agreement
between the a-priori standard and the calibrated configurations.


In [ ]:
# [cell 18]
# ---------------- plotting setup of the figures below -------------------------------------------------
import matplotlib.pyplot as plt
FIG_DIR = _OUT / "figures"   # figures, when a *_SAVE name is set
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#4a3aa7"]
INK, MUTED = "#0b0b0b", "#52514e"
DCOL = dict(zip(DESIGNS, SERIES))
# DES_LAB: the design labels of the a-priori run's record ([cell 3]); abbreviated by label where space is short
DES_ABBR = {"Dynamic": "Dyn.", "Predefined": "Pre.", "Control": "Ctl.", "Session": "Ses.", "Participant": "Part."}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#c9c8c3", "axes.labelcolor": INK, "text.color": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True, "grid.color": "#e8e7e3",
                     "grid.linewidth": 0.6, "axes.axisbelow": True, "legend.frameon": False, "font.size": 9})
# α vs matched null: total, and per trial removed (shown x 10^6)
QUANTS = {"overall": "Excess", "per-trial": "ExcessPerTrial"}
QSCALE = {"overall": 1.0, "per-trial": 1e6}
# Plateau figure: the exponent held constant while the other one varies; None = the calibrated pick's value.
PLATEAU_HOLD = {
    "cluster":     {"kmeans_exp_entropy_exponent": None, "kmeans_exp_rt_var_exponent": None},
    "mixture_gmm": {"kmeans_exp_entropy_exponent": None, "kmeans_exp_rt_var_exponent": None},
}


In [ ]:
# [cell 19]
# ---------------- 4a. Distribution of the overlap across ALL sweep configurations ----------
# One panel per arm, Jaccard (top row) and Dice / DSC (bottom row), with the a-priori standard over every exponent
# combination in the sweep, with the arm's pick marked. Shapiro-Wilk tests whether each distribution is normal
# (small p = not normal); the summary below reports median, IQR and range, which need no normality.
from scipy.stats import shapiro
_OV_ROWS = [("Jaccard", lambda a, b: len(a & b) / len(a | b) if (a | b) else np.nan),
            ("Dice (DSC)", lambda a, b: 2 * len(a & b) / (len(a) + len(b)) if (a or b) else np.nan)]
if APRIORI_CONFIG:
    _ref = get(APRIORI_CONFIG, "inattent_behav")
    _sw = []
    fig, axes = plt.subplots(len(_OV_ROWS), len(CM), figsize=(4.2 * len(CM), 3.4 * len(_OV_ROWS)), squeeze=False,
                             layout="constrained")
    for r, (ov_name, ov) in enumerate(_OV_ROWS):
        for k, (ax, m) in enumerate(zip(axes[r], CM)):
            v = pd.Series({c: ov(_ref, get(c, m)) for c in sorted(set(agg.loc[agg.method == m, "config"]))
                           if len(get(c, m)) >= MIN_OVERLAP_BLOCKS}, dtype=float).dropna()
            W, p = shapiro(v)
            pick = GOLD["a-priori inattent overlap"].get(m)
            ax.hist(v, bins=30, color=SERIES[k], edgecolor="white", linewidth=0.5)
            ax.axvline(v.median(), color=INK, lw=1.2, ls="--", label=f"median {v.median():.3f}")
            ax.axvspan(v.quantile(.25), v.quantile(.75), color=MUTED, alpha=0.12, label="IQR")
            if pick in v.index:
                ax.axvline(v[pick], color="#d62728", lw=1.6, label=f"pick {v[pick]:.3f}")
            ax.set_title(f"{short(m)}\nShapiro-Wilk W = {W:.3f}, p = {p:.2g}", fontweight="bold", fontsize=9.5, loc="left")
            ax.set_xlabel(f"{ov_name} with a-priori standard"); ax.legend(fontsize=7, frameon=False)
            _sw.append({"overlap": ov_name, "method": short(m), "n_configs": len(v), "W": W, "p": p,
                        "normal (p >= .05)": p >= .05, "skew": v.skew(), "kurtosis": v.kurt()})
        axes[r, 0].set_ylabel("configurations")
    fig.suptitle(f"Jaccard (top) and Dice (DSC, bottom) overlap with the a-priori standard across all sweep "
                 f"configurations ({DES_LAB['dynamic'].lower()} blocks, condition {SWEEP_COND})", fontweight="bold", x=0.01,
                 ha="left")
    plt.show()
    print("Shapiro-Wilk normality test per arm:")
    display(pd.DataFrame(_sw).set_index(["overlap", "method"]).round(4))


In [ ]:
# [cell 20]
# ---------------- 4b. Profiles of the calibrated picks ------------------------------------------------
# One column per arm at its calibrated config (the blocks of the analysis method, design "dynamic"). Top row: the classification
# features the arm was fitted on. Bottom row: the mean LVL_1 features of the same blocks, UNFOLDED
# (signed RT level, raw sliding-window variance, raw entropy), to show direction. Every feature is
# z-scored across blocks so the three share one axis; bars = profile mean z-score, error bars = SEM
# (the notebook's cluster-profile plot). The profile the speed rule flagged as inattentive is red.
L2_LABEL = {"adj_rt_level_block": "AdjRT\n(Block)", "exp_rt_var_block": "ExpVar\n(Block)",
            "exp_entropy_block": "ExpEnt\n(Block)", "exp_rt_var_block_gmm": "ExpVar\n(Block)",
            "exp_entropy_block_gmm": "ExpEnt\n(Block)"}
L1_LABEL = {"aggregated_residual": "AdjRT\n(Stim)", "local_residual_var": "AdjVar\n(win)",
            "rating_entropy_norm": "Ent\n(win)"}
_am = str(cfg_tab["cfg_analysis_method"].dropna().iloc[0]).strip("'\"")
_bcol = f"block_{_am}"
_l1 = pd.read_csv(f"{L1_PREFIX}_features.csv",
                  usecols=["id", "session", "condition", _bcol] + list(L1_LABEL))
_l1 = (_l1[(_l1.condition == SWEEP_COND) | (SWEEP_COND == "all")].groupby(["id", "session", _bcol])[list(L1_LABEL)].mean()
       .reset_index().rename(columns={_bcol: "block"}))

def _profile_file(cfg, arm):
    path = STAB_DIR / (re.sub(r"[^A-Za-z0-9_.-]", "_", cfg) + ".json")
    return path.with_name(path.stem + f"__{arm}_profiles.csv")

def _profile_panel(ax, df, cols, labels, flagged_profile, title=None):
    """Grouped bars as in the notebook's cluster-profile plot: one group per feature, one bar per
    profile, height = profile mean z-score, error bar = SEM (in z units)."""
    z = (df[cols] - df[cols].mean()) / df[cols].std(ddof=1)
    profs = sorted(df.profile.unique())
    width = 0.8 / len(profs)
    xs = np.arange(len(cols))
    others = iter(["#4a72b8", "#2f9e63", "#8c8c8c"])
    heights = {}
    for i, p in enumerate(profs):
        g = z[df.profile == p]
        mu, sem = g.mean(), g.std(ddof=1) / np.sqrt(len(g))
        red = p == flagged_profile
        heights[p] = (xs + (i - len(profs) / 2 + 0.5) * width, mu, len(g))
        ax.bar(heights[p][0], mu, width, yerr=sem.fillna(0), capsize=3, edgecolor="black", linewidth=0.5,
               color="#d62728" if red else next(others),
               label=f"profile {p}{' (flagged)' if red else ''}: n={len(g)} ({100 * len(g) / len(df):.0f}%)")
    # a tiny profile (e.g. a single outlying block) can sit far off the scale of the others:
    # keep the axis on the profiles with >= 5 blocks and label the clipped bars with their value
    big = [h for p, h in heights.items() if h[2] >= 5]
    if big and len(big) < len(heights):
        lim = 1.25 * max(np.nanmax(np.abs(h[1].to_numpy())) for h in big)
        ax.set_ylim(-lim, lim)
        for p, (x, mu, n) in heights.items():
            for xx, v in zip(x, mu):
                if abs(v) > lim:
                    ax.annotate(f"{v:.1f}", (xx, np.sign(v) * lim * 0.97), ha="center",
                                va="top" if v < 0 else "bottom", fontsize=8)
    ax.axhline(0, color="gray", ls="--", alpha=0.7, lw=0.8)
    ax.set_xticks(xs, [labels[c] for c in cols], fontsize=9)
    ax.grid(False)
    if title:
        ax.set_title(title, fontweight="bold", fontsize=10)

_W = GOLD["a-priori inattent overlap"]
fig, axes = plt.subplots(2, len(CM), figsize=(4.2 * len(CM), 7.2), squeeze=False, layout="constrained")
for j, m in enumerate(CM):
    arm, cfg = m.split("·")[0], _W[m]
    prof = pd.read_csv(_profile_file(cfg, arm))
    cols2 = ["adj_rt_level_block"] + ([f"exp_rt_var_block{'_gmm' * arm.startswith('mixture')}",
                                       f"exp_entropy_block{'_gmm' * arm.startswith('mixture')}"])
    fl = prof.loc[prof.flagged, "profile"]
    flagged_profile = fl.mode().iloc[0] if len(fl) else None
    title = (f"{short(m)}\nentropy exp {cfg_tab.loc[cfg, 'cfg_kmeans_exp_entropy_exponent']:g}, "
             f"RT-var exp {cfg_tab.loc[cfg, 'cfg_kmeans_exp_rt_var_exponent']:g}")
    _profile_panel(axes[0, j], prof, cols2, L2_LABEL, flagged_profile, title)
    _merged = prof.merge(_l1, on=["id", "session", "block"], how="inner")
    _profile_panel(axes[1, j], _merged, list(L1_LABEL), L1_LABEL, flagged_profile)
    axes[0, j].legend(fontsize=7.5, loc="best", frameon=False)
    if len(_merged) < len(prof):
        print(f"{short(m)}: {len(prof) - len(_merged)} blocks without LVL_1 features")
axes[0, 0].set_ylabel("classification features\nmean (z across blocks)", fontsize=9)
axes[1, 0].set_ylabel("LVL_1 features (unfolded)\nmean (z across blocks)", fontsize=9)
for k, ax in enumerate(axes.ravel()):
    ax.text(-0.08, 1.02, chr(65 + k), transform=ax.transAxes, fontsize=12, fontweight="bold", va="bottom")
fig.suptitle(f"Profiles of the calibrated configurations ({DES_LAB['dynamic'].lower()} blocks; mean z-score ± SEM; "
             "red = flagged profile)",
             fontweight="bold", x=0.01, ha="left")
PROFILE_SAVE = None    # a file name in FIG_DIR, e.g. "fig_lvl2_all_profiles.pdf"
if PROFILE_SAVE:
    fig.savefig(FIG_DIR / PROFILE_SAVE, dpi=300, bbox_inches="tight")
plt.show()

*[cell 21]*

### 4c. Is the calibration a plateau? Overlap (Jaccard or Dice, `OVERLAP_STAT`) vs one exponent, the other held constant

Left: overlap with the a-priori standard as `kmeans_exp_entropy_exponent` varies; right: as `kmeans_exp_rt_var_exponent` varies. The bold line holds
the other exponent at `PLATEAU_HOLD` (default: the calibrated pick's value); faint lines are every other held value,
so a flat bold line inside a tight faint band is a real plateau, while a bold line that moves on its own is not.
Numbers on the bold line are % of blocks flagged — overlap that only rises with volume is dose, not calibration.

In [ ]:
# [cell 22]
if APRIORI_CONFIG:
    _ref = get(APRIORI_CONFIG, "inattent_behav")
    JT = []
    for m in CM:
        rm = m.split("·")[0]
        for c in sorted(set(agg.loc[agg.method == m, "config"])):
            s = get(c, m)
            hb = h2[(h2.config == c) & (h2.method == m) & (h2.Design == "dynamic")]
            JT.append({"method": m, "rm": rm, "config": c, "J": jac(_ref, s) if len(s) else np.nan,
                       "kmeans_exp_entropy_exponent": cfg_tab.loc[c, "cfg_kmeans_exp_entropy_exponent"], "kmeans_exp_rt_var_exponent": cfg_tab.loc[c, "cfg_kmeans_exp_rt_var_exponent"],
                       "pct_blocks": hb.pct_blocks_flagged.iloc[0] if len(hb) else np.nan})
    JT = pd.DataFrame(JT)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), layout="constrained")
    for ax, (x, held) in zip(axes, (("kmeans_exp_entropy_exponent", "kmeans_exp_rt_var_exponent"), ("kmeans_exp_rt_var_exponent", "kmeans_exp_entropy_exponent"))):
        for k, m in enumerate(CM):
            t = JT[JT.method == m]
            rm = t.rm.iloc[0]
            pick = GOLD["a-priori inattent overlap"].get(m)
            hv = PLATEAU_HOLD.get(family(rm), {}).get(held)
            hv = cfg_tab.loc[pick, f"cfg_{held}"] if hv is None and pick else hv
            col = SERIES[k]
            for v, g in t.groupby(held):
                g = g.sort_values(x)
                if v != hv:
                    ax.plot(g[x], g.J, color=col, lw=0.8, alpha=0.18)
            g = t[t[held] == hv].sort_values(x)
            ax.plot(g[x], g.J, color=col, lw=2.4, marker="o", ms=5, label=f"{short(m)} ({held} = {hv:g})")
            for _, r in g.iterrows():
                ax.annotate(f"{r.pct_blocks:.0f}%", (r[x], r.J), textcoords="offset points", xytext=(0, 6),
                            ha="center", fontsize=6.5, color=MUTED)
            if pick is not None and pick in set(g.config):
                pr = g[g.config == pick].iloc[0]
                ax.scatter(pr[x], pr.J, s=140, facecolor="none", edgecolor=INK, linewidth=1.5, zorder=5)
        ax.set_xlabel(x); ax.set_ylabel(f"{OV_NAME} with a-priori standard ({DES_LAB['dynamic'].lower()} blocks)")
        ax.set_title(f"overlap vs {x}", fontweight="bold", loc="left", fontsize=10)
        ax.legend(fontsize=8, loc="best")
    plt.show()

*[cell 23]*

### 4d. Removal outcome by design

**Overall** (α gain beyond matched-volume random removal) then **per-trial** (the same, per trial removed, × 10⁻⁶).
Bars = mean over scales with SE; scatter = cost vs benefit. Set `PUB_SAVE` to write a 300-dpi file.

In [ ]:
# [cell 24]
import matplotlib.transforms as _mtr


def _bbox_overlap(a, b):
    dx = min(a.x1, b.x1) - max(a.x0, b.x0)
    dy = min(a.y1, b.y1) - max(a.y0, b.y0)
    return dx * dy if dx > 0 and dy > 0 else 0.0


def place_labels(ax, xs, ys, labels, fontsize=6.5, color="#666666"):
    """Annotate without overlaps. Searches rings of candidate offsets outward from each
    point and keeps the first placement that collides with nothing already placed; when
    a label has to sit far from its marker a thin leader line is drawn so the pairing
    stays unambiguous. When every ring is blocked, the least-overlapping candidate is
    kept."""
    fig = ax.figure
    fig.canvas.draw()
    rend = fig.canvas.get_renderer()
    cands = [(0, 0, 6, 3, "left", "bottom")]
    for rad in (8, 16, 26, 38, 52, 68):
        for ang in range(0, 360, 30):
            a = np.deg2rad(ang)
            dx, dy = rad * np.cos(a), rad * np.sin(a)
            ha = "left" if dx > 2 else ("right" if dx < -2 else "center")
            va = "bottom" if dy > 2 else ("top" if dy < -2 else "center")
            cands.append((rad, ang, dx, dy, ha, va))
    taken = []
    for x, y in zip(xs, ys):                      # markers are obstacles too
        px, py = ax.transData.transform((x, y))
        taken.append(_mtr.Bbox.from_bounds(px - 5, py - 5, 10, 10))
    for i in np.argsort(ys)[::-1]:                # densest region first
        best = None
        for rad, ang, dx, dy, ha, va in cands:
            t = ax.annotate(labels[i], (xs[i], ys[i]), textcoords="offset points",
                            xytext=(dx, dy), fontsize=fontsize, color=color,
                            ha=ha, va=va, annotation_clip=False,
                            arrowprops=dict(arrowstyle="-", lw=0.4, color="#b6b5b0",
                                            shrinkA=0, shrinkB=2) if rad >= 26 else None)
            bb = t.get_window_extent(rend).expanded(1.04, 1.20)
            ov = sum(_bbox_overlap(bb, o) for o in taken)
            if best is None or ov < best[1]:
                if best is not None:
                    best[0].remove()
                best = (t, ov, bb)
            else:
                t.remove()
            if ov == 0:
                break
        taken.append(best[2])


PUB_ROWS = [(g, q) for g in GOLD for q in ("overall", "per-trial")]
PUB_DES = [d for d in DESIGNS if d in set(df.Design)]   # the designs of the sweep's runs present in its results
PUB_SAVE = None    # a file name in FIG_DIR, e.g. "fig_lvl2_all_designs.pdf"

QLAB = {"overall": "α gain vs matched null (Δ)", "per-trial": "α gain vs null per trial removed (×10⁻⁶)"}

def set_rows(W):
    out = []
    for m, cfg in W.items():
        raw = df[(df.config == cfg) & (df.method == m)]
        for d in PUB_DES:
            r = raw[raw.Design == d]
            rec = {"method": m, "Design": d, "pct": r.Pct_Removed.mean()}
            for tag, col in QUANTS.items():
                v = QSCALE[tag] * r[f"Alpha_{col}"].dropna()
                rec[tag] = v.mean()
                rec[f"{tag}_se"] = v.std(ddof=1) / np.sqrt(len(v)) if len(v) > 1 else np.nan
            if len(r):
                out.append(rec)
    return pd.DataFrame(out)

FIG_DATA = {name: set_rows(W) for name, W in GOLD.items()}

# the scatter panels carry 20 labels each, so they get the wider half
fig, axes = plt.subplots(len(PUB_ROWS), 2, figsize=(17, 4.2 * len(PUB_ROWS)), squeeze=False,
                         gridspec_kw={"width_ratios": [1, 1.45]}, layout="constrained")
for r, (name, q) in enumerate(PUB_ROWS):
    t, ms = FIG_DATA[name], list(GOLD[name])
    ax_b, ax_s = axes[r, 0], axes[r, 1]
    w = 0.8 / len(PUB_DES)
    for j, d in enumerate(PUB_DES):
        s = t[t.Design == d].set_index("method").reindex(ms)
        ax_b.bar(np.arange(len(ms)) + (j - (len(PUB_DES) - 1) / 2) * w, s[q], width=w * 0.92,
                 yerr=s[f"{q}_se"], capsize=2.5, color=DCOL[d], error_kw=dict(lw=0.9, ecolor=MUTED))
    ax_b.axhline(0, color="#9b9a95", lw=0.8)
    ax_b.set_xticks(range(len(ms)), [short(m) for m in ms], fontsize=8)
    ax_b.set_ylabel(QLAB[q]); ax_b.grid(axis="x", visible=False)
    ax_b.set_title(f"{name} — {q}", fontweight="bold", loc="left", fontsize=9)

    for d in PUB_DES:
        s = t[t.Design == d]
        ax_s.scatter(s.pct, s[q], s=70, color=DCOL[d], edgecolor="white", linewidth=1.4, zorder=3)
    # margins first so the axis limits are final before labels are measured
    ax_s.axhline(0, color="#9b9a95", lw=0.8, ls="--"); ax_s.margins(x=0.18, y=0.15)
    place_labels(ax_s, t["pct"].values, t[q].values,
                 [f"{short(m)} ({DES_ABBR.get(DES_LAB[d], DES_LAB[d])})" for m, d in zip(t["method"], t["Design"])],
                 fontsize=6.5, color=MUTED)
    ax_s.set_xlabel("% data (trials) removed"); ax_s.set_ylabel(QLAB[q])

    for k, ax in enumerate((ax_b, ax_s)):
        ax.text(-0.1, 1.06, chr(65 + 2 * r + k), transform=ax.transAxes, fontsize=12, fontweight="bold", va="bottom")

fig.legend([plt.Rectangle((0, 0), 1, 1, color=DCOL[d]) for d in PUB_DES], [DES_LAB[d] for d in PUB_DES],
           loc="outside upper center", ncol=len(PUB_DES), fontsize=9)
if PUB_SAVE:
    fig.savefig(FIG_DIR / PUB_SAVE, dpi=300, bbox_inches="tight")
plt.show()

*[cell 25]*

### 4e. Method agreement

Overlap (Jaccard or Dice, `OVERLAP_STAT`) of flagged blocks between the a-priori standard and the calibrated cluster / GMM configurations;
one panel per block design of the sweep's runs: the analysis method's blocks (`dynamic`), and in dynamic segmentation
the control blocks (`control`).

In [ ]:
# [cell 26]
JAC_ROWS = list(GOLD)
JAC_SAVE = None    # a file name in FIG_DIR, e.g. "fig_lvl2_all_overlap.pdf"

JAC_DES = [d for d in ("dynamic", "control") if d in DESIGNS]   # the block designs of the sweep's runs
fig, axes = plt.subplots(len(JAC_ROWS), len(JAC_DES), figsize=(5.25 * len(JAC_DES), 3.5 * len(JAC_ROWS)), squeeze=False,
                         layout="constrained")
for r, name in enumerate(JAC_ROWS):
    W = GOLD[name]; ms = list(W)
    for c, d in enumerate(JAC_DES):
        ax = axes[r, c]
        sets = {m: get(W[m], m, d) for m in ms}
        mat = np.array([[jac(sets[a], sets[b]) for b in ms] for a in ms], dtype=float)
        im = ax.imshow(mat, cmap="Blues", vmin=0, vmax=1); ax.grid(False)
        for i_ in range(len(ms)):
            for j_ in range(len(ms)):
                if np.isfinite(mat[i_, j_]):
                    ax.text(j_, i_, f"{mat[i_, j_]:.2f}", ha="center", va="center", fontsize=8,
                            color="white" if mat[i_, j_] > 0.6 else INK)
        ax.set_xticks(range(len(ms)), [f"{short(m)}\n(n={len(sets[m])})" for m in ms], fontsize=7.5)
        ax.set_yticks(range(len(ms)), [short(m) for m in ms], fontsize=7.5)
        ax.set_title(f"{name} — {DES_LAB[d]}", fontweight="bold", loc="left", fontsize=8.5)
        ax.text(-0.18, 1.06, chr(65 + len(JAC_DES) * r + c), transform=ax.transAxes, fontsize=12, fontweight="bold",
                va="bottom")
fig.colorbar(im, ax=axes, shrink=0.5, label=f"{OV_NAME} overlap")
if JAC_SAVE:
    fig.savefig(FIG_DIR / JAC_SAVE, dpi=300, bbox_inches="tight")
plt.show()